# Camada Gold Modelo Analítico (Star Schema)

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ler a Silver limpa, integrar as fontes e modelar o star schema para consumo do Power BI.

**Modelo:**
- `dim_uf` — dimensão de estados
- `dim_municipio` — dimensão de municípios
- `dim_tempo` — dimensão de anos
- `fato_indicador` — fato central: taxa realizada vs. meta, particionado por ano

**Princípio da camada:** a Gold é a única camada que o Power BI acessa — sem transformação pesada no Power Query, só leitura direta.

## 1. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault**, mesmo padrão da Bronze e da Silver.

In [41]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# ============================================================
from notebookutils import mssparkutils

CONTA_STORAGE = "stalfalfabetizacao"

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")

StatementMeta(sparkpool, 24, 18, Finished, Available, Finished, False)

Conexão configurada com sucesso!


## 2. Leitura da Camada Silver

A Gold nunca lê da Bronze — sempre parte da Silver já tratada e padronizada.

In [42]:
# ============================================================
# Leitura das tabelas Silver
# A Gold nunca lê da Bronze — sempre parte da Silver limpa
# ============================================================

df_uf            = spark.read.parquet(CAMINHO_SILVER + "uf/")
df_municipio     = spark.read.parquet(CAMINHO_SILVER + "municipio/")
df_meta_brasil   = spark.read.parquet(CAMINHO_SILVER + "meta_brasil/")
df_meta_uf       = spark.read.parquet(CAMINHO_SILVER + "meta_por_uf/")
df_meta_municipio = spark.read.parquet(CAMINHO_SILVER + "meta_por_municipio/")
df_ts_municipio  = spark.read.parquet(CAMINHO_SILVER + "ts_municipio/")
df_ts_estado     = spark.read.parquet(CAMINHO_SILVER + "ts_estado/")

print("Silver lida com sucesso!")
print(f"  municipio:      {df_municipio.count()} linhas")
print(f"  meta_municipio: {df_meta_municipio.count()} linhas")
print(f"  ts_municipio:   {df_ts_municipio.count()} linhas")

StatementMeta(sparkpool, 24, 19, Finished, Available, Finished, False)

Silver lida com sucesso!
  municipio:      23995 linhas
  meta_municipio: 10704 linhas
  ts_municipio:   12416 linhas


## 3. Construção do Star Schema

Modelagem dimensional: `dim_uf`, `dim_municipio`, `dim_tempo` e o fato central `fato_indicador`, que integra o realizado (`municipio`) com a meta (`meta_por_municipio`) via join.

## 3.1 Correção — Mapeamento de `rede` antes do join

**Problema identificado:** o join entre `municipio` e `meta_por_municipio` usava `rede` como chave, mas as tabelas tratam essa coluna de forma incompatível — `municipio.rede` vem como código numérico (0, 1, 2, 3, 4, 5, 6), enquanto `meta_por_municipio.rede` vem como texto (ex: `"Municipal"`). Resultado: nenhuma linha casava, e a coluna `meta_alfabetizacao_2024` ficava 100% em branco no `fato_indicador`.

**Correção:** mapeamento do código para o nome, conforme o dicionário oficial da Base dos Dados (`basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`, consultado via BigQuery).

**Consequência esperada:** como `meta_por_municipio` só possui registros de rede `"Municipal"`, o join continuará gerando `NULL` nas colunas de meta para as linhas de `municipio` com outras redes (Federal, Estadual etc.) — isso é comportamento correto do negócio, não bug.

In [43]:
# ============================================================
# GOLD — Star Schema
# ============================================================
from pyspark.sql.functions import col, when

# ── DIM_UF ──────────────────────────────────────────────────
dim_uf = df_uf \
    .select("sigla_uf") \
    .dropDuplicates() \
    .orderBy("sigla_uf")

# ── DIM_MUNICÍPIO ────────────────────────────────────────────
dim_municipio = df_municipio \
    .select("id_municipio") \
    .dropDuplicates(["id_municipio"]) \
    .join(
        df_ts_municipio.select(
            col("id_municipio"),
            col("sigla_uf")
        ).dropDuplicates(["id_municipio"]),
        on="id_municipio",
        how="left"
    ) \
    .orderBy("id_municipio")

# ── DIM_TEMPO ────────────────────────────────────────────────
dim_tempo = df_municipio \
    .select("ano") \
    .dropDuplicates() \
    .orderBy("ano")

# ── FATO_INDICADOR ───────────────────────────────────────────
# Mapeia código de rede (municipio) para nome, conforme dicionário oficial
# da Base dos Dados (basedosdados.br_inep_avaliacao_alfabetizacao.dicionario)
# — precisa ser feito ANTES do join, senão o join continua falhando
df_mun = df_municipio.withColumn(
    "rede",
    when(col("rede") == 0, "Total (Federal, Estadual, Municipal)")
    .when(col("rede") == 1, "Federal")
    .when(col("rede") == 2, "Estadual")
    .when(col("rede") == 3, "Municipal")
    .when(col("rede") == 4, "Privada")
    .when(col("rede") == 5, "Pública (Estadual e Municipal)")
    .when(col("rede") == 6, "Pública (Federal, Estadual e Municipal)")
    .otherwise("Não classificada")
).alias("mun")

df_meta = df_meta_municipio.alias("meta")

fato_indicador = df_mun \
    .join(
        df_meta,
        on=["id_municipio", "ano", "rede"],
        how="left"
    ) \
    .select(
        col("mun.id_municipio"),
        col("mun.ano"),
        col("mun.rede"),
        col("mun.taxa_alfabetizacao").alias("taxa_realizada"),
        col("mun.media_portugues"),
        col("meta.taxa_alfabetizacao").alias("taxa_meta_referencia"),
        col("meta.meta_alfabetizacao_2024"),
        col("meta.meta_alfabetizacao_2025"),
        col("meta.meta_alfabetizacao_2026"),
        col("meta.meta_alfabetizacao_2027"),
        col("meta.meta_alfabetizacao_2028"),
        col("meta.meta_alfabetizacao_2029"),
        col("meta.meta_alfabetizacao_2030"),
        col("meta.nivel_alfabetizacao"),
        col("meta.percentual_participacao")
    ) \
    .dropDuplicates()

# Confirma
print("Star schema construído:")
print(f"  dim_uf:         {dim_uf.count()} linhas")
print(f"  dim_municipio:  {dim_municipio.count()} linhas")
print(f"  dim_tempo:      {dim_tempo.count()} linhas")
print(f"  fato_indicador: {fato_indicador.count()} linhas")

StatementMeta(sparkpool, 24, 20, Finished, Available, Finished, False)

Star schema construído:
  dim_uf:         25 linhas
  dim_municipio:  5550 linhas
  dim_tempo:      2 linhas
  fato_indicador: 23995 linhas


## 4. Fatos Adicionais Meta por UF e Meta Brasil

O PDF exige a integração das 3 fontes de meta. Cada uma se compara com o realizado da sua própria granularidade (evita misturar níveis num fato só, o que quebraria agregações no Power BI):

- `fato_indicador` → município × meta_por_municipio (rede Municipal)
- `fato_indicador_uf` → uf × meta_por_uf (rede Pública)
- `fato_indicador_brasil` → meta_brasil (nível nacional)

**Decisão documentada:** `meta_alfabetizacao_uf` usa a categoria `"Pública"`, enquanto `uf` usa códigos. Pelo dicionário oficial, o código `5 = "Pública (Estadual e Municipal)"` é o equivalente semântico — o mapeamento abaixo faz essa correspondência explícita (validada via BigQuery em 07/2026).

In [44]:
# ============================================================
# FATO_INDICADOR_UF — realizado (uf) vs meta (meta_por_uf)
# Chave de negócio: a meta "Pública" da UF corresponde ao
# realizado da rede 5 ("Pública Estadual e Municipal"),
# conforme dicionário oficial validado via BigQuery
# ============================================================
from pyspark.sql.functions import col, when

# Mapeia código de rede da uf para o texto usado na meta_uf
df_uf_mapeada = df_uf.withColumn(
    "rede",
    when(col("rede") == 5, "Pública")          # equivalência validada via dicionário
    .when(col("rede") == 0, "Total")
    .when(col("rede") == 2, "Estadual")
    .when(col("rede") == 3, "Municipal")
    .otherwise("Não classificada")
).alias("ufr")

df_meta_uf_alias = df_meta_uf.alias("metauf")

fato_indicador_uf = df_uf_mapeada \
    .join(
        df_meta_uf_alias,
        on=["sigla_uf", "ano", "rede"],
        how="left"
    ) \
    .select(
        col("ufr.sigla_uf"),
        col("ufr.ano"),
        col("ufr.rede"),
        col("ufr.taxa_alfabetizacao").alias("taxa_realizada"),
        col("ufr.media_portugues"),
        col("metauf.taxa_alfabetizacao").alias("taxa_meta_referencia"),
        col("metauf.meta_alfabetizacao_2024"),
        col("metauf.meta_alfabetizacao_2025"),
        col("metauf.meta_alfabetizacao_2026"),
        col("metauf.meta_alfabetizacao_2027"),
        col("metauf.meta_alfabetizacao_2028"),
        col("metauf.meta_alfabetizacao_2029"),
        col("metauf.meta_alfabetizacao_2030").cast("double").alias("meta_alfabetizacao_2030"),
        col("metauf.percentual_participacao")
    ) \
    .dropDuplicates()

# ============================================================
# FATO_INDICADOR_BRASIL — nível nacional
# A própria meta_brasil traz o dado nacional por ano/rede
# ============================================================
fato_indicador_brasil = df_meta_brasil.dropDuplicates()

print("Fatos adicionais construídos:")
print(f"  fato_indicador_uf:     {fato_indicador_uf.count()} linhas")
print(f"  fato_indicador_brasil: {fato_indicador_brasil.count()} linhas")

StatementMeta(sparkpool, 24, 21, Finished, Available, Finished, False)

Fatos adicionais construídos:
  fato_indicador_uf:     145 linhas
  fato_indicador_brasil: 3 linhas


In [45]:
# ============================================================
# Gravação da camada Gold em formato Parquet
# Particionamento por "ano" na fato_indicador —
# queries filtrando por ano leem só a partição necessária
# (menos dados lidos = menor custo = FinOps)
# ============================================================

dim_uf.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_uf/"
)
dim_municipio.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_municipio/"
)
dim_tempo.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_tempo/"
)

# Fato particionado por ano — padrão FinOps
fato_indicador.write \
    .mode("overwrite") \
    .partitionBy("ano") \
    .parquet(CAMINHO_GOLD + "fato_indicador/")

# Fatos adicionais — meta por UF e Brasil
fato_indicador_uf.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "fato_indicador_uf/"
)
fato_indicador_brasil.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "fato_indicador_brasil/"
)

print("Gold gravada com sucesso!")
print("Tabelas disponíveis:")
print(f"  {CAMINHO_GOLD}dim_uf/")
print(f"  {CAMINHO_GOLD}dim_municipio/")
print(f"  {CAMINHO_GOLD}dim_tempo/")
print(f"  {CAMINHO_GOLD}fato_indicador/  (particionada por ano)")
print(f"  {CAMINHO_GOLD}fato_indicador_uf/")
print(f"  {CAMINHO_GOLD}fato_indicador_brasil/")

StatementMeta(sparkpool, 24, 22, Finished, Available, Finished, False)

Gold gravada com sucesso!
Tabelas disponíveis:
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_uf/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_municipio/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_tempo/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/fato_indicador/  (particionada por ano)
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/fato_indicador_uf/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/fato_indicador_brasil/


### Validação da Gold

In [46]:
# ============================================================
# Validação da Gold
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.functions import col
# Lê de volta do storage pra confirmar que gravou certo
fato_lido = spark.read.parquet(CAMINHO_GOLD + "fato_indicador/")

print("=== VALIDAÇÃO FATO_INDICADOR ===")
print(f"Total linhas:     {fato_lido.count()}")
print(f"Nulos id_municipio: {fato_lido.filter(col('id_municipio').isNull()).count()}")
print(f"Nulos taxa_realizada: {fato_lido.filter(col('taxa_realizada').isNull()).count()}")
print(f"Anos disponíveis:")
fato_lido.select("ano").dropDuplicates().orderBy("ano").show()
print(f"Taxa min/max:")
fato_lido.select(
    F.min("taxa_realizada"),
    F.max("taxa_realizada")
).show()

StatementMeta(sparkpool, 24, 23, Finished, Available, Finished, False)

=== VALIDAÇÃO FATO_INDICADOR ===
Total linhas:     23995
Nulos id_municipio: 0
Nulos taxa_realizada: 0
Anos disponíveis:
+----+
| ano|
+----+
|2023|
|2024|
+----+

Taxa min/max:
+-------------------+-------------------+
|min(taxa_realizada)|max(taxa_realizada)|
+-------------------+-------------------+
|               2.12|              100.0|
+-------------------+-------------------+

